# Gold Biorecovery from E-Waste — Coupled Model

**One notebook over three models.** Crushed printed circuit boards in, recovered
metallic gold out.

| Model | Source document | What it covers |
|---|---|---|
| Bioleaching | `Bioleaching_Models_Sequential.docx` | three vessels: PCB → dissolved Au(S₂O₃)₂³⁻ |
| GolB display | `GolB_model_writeup.docx` | how many gold-binding sites a cell carries |
| Recovery chain | `model.md` v3.0 | capture → elution → reduction → recovery |

The three documents are reproduced verbatim in `docs/MODEL.md` and
`docs/MODEL.docx`. Nothing was corrected. Discrepancies found while combining
them are listed in `docs/ISSUES.md`.

This notebook holds narrative and figures only. Every equation lives in `src/`
and every number lives in `params/`. If you find yourself writing a `def` here,
it belongs in `src/`.

## Validation status — read first

`model.md` §0 opens with a statement that governs everything below:

> No part of this model has been checked against a measurement of this system.

That was written about the recovery chain. It applies at least as strongly to
the bioleaching and display models, whose parameter tables read *calibrate*,
*unmeasured*, *fitted*, *assumed*, *placeholder*.

**The deliverable is a ranked measurement priority list, not a yield prediction.**

### Recovery review update, 6 September 2026

The current recovery report is [RECOVERY.md](docs/RECOVERY.md), with a recovery-only
wiki proposal in [WIKI_RECOVERY.md](docs/WIKI_RECOVERY.md). Issues 84–89 record the
corrections. Nominal reduction is a proxy scenario; use the full kinetic-range
figure for uncertainty. Historical PRCC outputs are not validated current results.
Protein dilution now follows actual logistic growth, so previous harvest and
yield recommendations are withdrawn pending calibration. Synthetic test agreement
verifies code, not the recovery chemistry.


## The chain, and the two handovers

```
  crushed PCB
      │
      │  BIOLEACHING — three vessels in series
      │    1  culture      E. coli build TetH, harvested and purified
      │    2  conversion   TetH turns tetrathionate into thiosulfate
      │    3  leach        thiosulfate dissolves Au, and Cu
      ▼
  leachate:  [Au(S2O3)2 3-],  [Cu(I)],  [Cu(II)],  [S4O6 2-]
      │
      │  ══ HANDOVER 1 ══  Au_feed      interfaces.au_feed_from_leach
      ▼
  CAPTURE on GolB-displaying E. coli
      ▲
      │  ══ HANDOVER 2 ══  q_max        interfaces.qmax_from_expression
      │
  GOLB DISPLAY — mRNA → cytoplasmic protein → surface protein → active sites

  RECOVERY continues:  capture → elution → reduction → recovery → eta_total
```

The three models are not three topics. Two quantities connect them, and each is
a quantity one document declares missing and another supplies.

**Handover 1 — `Au_feed`.** The recovery-chain document §10: *"`Au_feed` has no
upstream model though it dominates V1."* Bioleaching computes exactly that.

**Handover 2 — `q_max`.** The recovery-chain document §9 ranks `q_max`
measurement priority 2, noting it is *"being addressed by the secretion
model."* The display model computes the site count that sets it.

## Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))

from src import expression, interfaces, leach, params, recovery_chain

plt.rcParams.update({"figure.dpi": 110, "font.size": 9,
                     "axes.grid": True, "grid.alpha": 0.3})

P_EXPR = params.values(params.load("expression"))
P_CHAIN = params.load("recovery_chain")

print("parameter files:", [p.stem for p in sorted(params.PARAMS_DIR.glob("*.yaml"))])

### What is not measured

`model.md` §0 records a track record worth repeating before any number is read:

> Every parameter assumed without a source, then sourced, was wrong.

So the first thing the notebook prints is the list of parameters that are still
fitted, assumed, placeholder, or awaiting calibration.

In [ ]:
for part in ("leach", "expression", "recovery_chain"):
    flagged = params.unsourced(part)
    print(f"\n{part}  ({len(flagged)} not sourced)")
    for symbol, status in flagged:
        print(f"   {symbol:<28} {status[:70]}")

---

# Bioleaching — three vessels in series

*Source: `docs/source/sequential_2026-08-12/`. Code: `src/leach.py`.
Every equation, with its annotation: `docs/EQUATIONS.md`.*

| Vessel | Contents | What it makes |
|---|---|---|
| **culture** | *E. coli* culture | the enzyme: TetH, harvested and purified |
| **conversion** | purified TetH + tetrathionate broth | the leachant: thiosulfate liquor |
| **leach** | thiosulfate liquor + crushed board | dissolved gold, and unavoidably copper |

**The culture vessel.** Solved as algebra, not integrated. Enzyme is built by
the cells, broken down by proteases, and diluted by division:

$$\frac{d[E]}{dt} = k_{syn} X - (k_{deg} + \mu)[E]
\qquad\Rightarrow\qquad
[E]_{culture} = \frac{k_{syn}X}{k_{deg}+\mu}$$

It sizes the culture and then leaves the model. There are no cells downstream.

**The conversion vessel.** No cells, so the enzyme is an input. No copper, so
nothing destroys the product. Every loss term vanishes:

$$\frac{d[S_4O_6^{2-}]}{dt} = -\frac{k_{cat}E_2[S_4O_6^{2-}]}{K_{S4O6}+[S_4O_6^{2-}]}
\qquad
\frac{d[S_2O_3^{2-}]}{dt} = +Y\,\frac{k_{cat}E_2[S_4O_6^{2-}]}{K_{S4O6}+[S_4O_6^{2-}]}$$

Since $Y = 1$, total sulfur is conserved. The whole broth charge converts and
banks, which is the point of the design.

**The leach vessel.** No enzyme, so thiosulfate is a fixed budget. Gold is
oxidised by oxygen and by copper(II) in parallel:

$$R_{Au} = a_{Au}(t)\,\frac{[S_2O_3^{2-}]}{K_{S2O3}+[S_2O_3^{2-}]}
\left\{ k_{Au,O_2}\frac{[O_2]}{K_{O_2}+[O_2]}
+ k_{Au,Cu}\frac{[Cu(II)]}{K_{CuII}+[Cu(II)]} \right\}$$

**The two geometries differ, and this is the interesting part.** Gold on a PCB
is a plated film, so it thins at constant face area:
$a_{Au} = m_{Au}/(\rho_{Au} t_{film})$, independent of how finely the board is
crushed. Copper is bulk metal and erodes inward, so
$a_{Cu}(t) = a_{Cu}(0)(n/n_0)^{2/3}$ and crush size does matter.

In [ ]:
# Geometry: gold area is set by plating thickness, not by crush size.
m_Au, rho_Au = 1e-4, 19300.0           # kg, kg/m3
for t_film in (0.05e-6, 0.5e-6, 1.0e-6):
    A = leach.exposed_gold_area(m_Au, rho_Au, t_film)
    print(f"t_film = {t_film*1e6:.2f} um  ->  a_Au = {A*1e4:7.2f} cm2")

print()
m_Cu, rho_Cu = 1e-2, 8960.0
for d in (2e-3, 4e-3, 6e-3):
    A = leach.initial_copper_area(m_Cu, rho_Cu, d)
    print(f"crush d = {d*1e3:.0f} mm    ->  a_Cu(0) = {A*1e4:7.2f} cm2")

### A demonstration run of the leach vessel

**These rate constants are not the model's values.** They are deliberately
artificial, so that one effect can be isolated at a time. **Read the shapes,
not the numbers.** The real parameter set is `leach.parameters()`, used further
down, and every quotable leaching number in this notebook comes from there.

Only the leach vessel carries the full chemistry, so that is what this exercises.
Inventories are in µmol; see `docs/ISSUES.md` item 4 for why that choice
matters.

In [ ]:
M_AU, M_CU = 197e-3, 63.55e-3          # kg/mol

demo = leach.refresh(dict(
    K_S4O6=500.0, k_par=1e-5, Y=1.0, k_cat=125.0,      # tetrathionate
    E_stage3=0.0, carry_enzyme=False,                  # no enzyme in the leach
    V_R=1.0, O2=250.0, t_stage3=600.0,                 # reactor
    m_Au=1e-4, rho_Au=19300.0, t_film=1e-6,            # gold
    k_Au_O2=11.0, k_Au_Cu=11.0e3, K_O2=50.0, K_S2O3=5.0e4, K_CuII=100.0,
    m_Cu=1e-2, rho_Cu=8960.0, d=4e-3,                  # copper
    k_Cu_O2=1.0, k_Cu_Cu=1.0, k_ox_Cu=1e-6,            # k_par is shared, ISSUES 16
    pH_on=False, pH0=7.0, beta=1e5,
    M_Au=M_AU, M_Cu=M_CU,
))

# Seeded with copper already in solution, so the Cu(II) route is alive.
y0 = [0.0, 1e4, 1e5, demo["n_Au0"], 0.0, demo["n_Cu0"], 500.0, 500.0, 0.0]
t, tr = leach.simulate_stage3(demo, 1e4, 1e5, t_end=600.0, y0=y0, model="B")

print(f"integrated to t = {t[-1]:.0f} min of 600")
print(f"gold dissolved: {demo['n_Au0'] - tr['n_Au'][-1]:.2f} umol of "
      f"{demo['n_Au0']:.1f} ({100*(1 - tr['n_Au'][-1]/demo['n_Au0']):.1f}%)")
print(f"leachate gold : {tr['Au_complex'][-1]:.3f} uM")

### Where the enzyme capacity comes from, from 2026-08-11

*E. coli* runs the whole chain and makes the thiosulfate itself. It has no
native tetrathionate metabolism, so the enzyme capacity cannot come from a
growth yield. It is counted directly instead, as `k_cat * [TetH]`, with
`[TetH]` from the GolB display cascade pointed at TetH. See `docs/ISSUES.md`
item 30.


In [ ]:
P_EXPR_TETH = params.values(params.load("expression"))
_, _, _, Ps, _ = expression.simulate(P_EXPR_TETH, t_end=2000.0)
teth_per_cell = P_EXPR_TETH["f"] * Ps[-1]

CAP = params.load("leach")["enzyme_capacity"]
K_CAT = params.load("leach")["tetrathionate"]["k_cat"]["value"]
cells_per_g = CAP["cells_per_g"]["value"]

print(f"active TetH per cell      : {teth_per_cell:,.0f} molecules")
print(f"k_cat                     : {K_CAT} /min")
print()
print(f"{'biomass g/L':>12}  {'[TetH] uM':>10}  {'Vmax umol/L/min':>16}")
for B in (0.1, 1.0, 4.0, 10.0, 30.0):
    v = leach.vmax_from_expression(teth_per_cell, B * cells_per_g, K_CAT)
    mark = "  <- in use" if B == CAP["biomass"]["value"] else ""
    print(f"{B:12.1f}  {v/K_CAT:10.4f}  {v:16.2f}{mark}")

print()
print("The retired A. ferrooxidans route claimed 14.3 at 0.1 g/L.")
print("E. coli needs 4.0 g/L for the same number: about 40x weaker per gram.")


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(11, 3.2))

ax[0].plot(t, tr["Vmax"])
ax[0].set(xlabel="time (min)", ylabel="Vmax  (umol/L/min)", title="(a) TetH capacity")

ax[1].plot(t, tr["S4O6"], label="S$_4$O$_6^{2-}$")
ax[1].plot(t, tr["S2O3"], label="S$_2$O$_3^{2-}$")
ax[1].set(xlabel="time (min)", ylabel="umol/L", title="(b) sulfur species")
ax[1].legend()

ax[2].plot(t, tr["Au_complex"], label="Au(S$_2$O$_3$)$_2^{3-}$")
ax[2].set(xlabel="time (min)", ylabel="uM", title="(c) leachate gold")
ax[2].legend()

fig.suptitle("Leach vessel demonstration run — placeholder rate constants", y=1.04)
fig.tight_layout()

**A structural warning visible in this run.** Copper supplies the fast gold
oxidant, and it was also the part that did not close. The write-up subtracts the
whole gold rate from Cu(II), including its oxygen pathway, which consumes no
copper. Starting from a copper-free solution, Cu(II) then has a sink and no
source and goes negative.

Applying each balanced reaction's own coefficients, per pathway, removes it.
Both are kept so the disagreement stays visible: `stoichiometry="source"`
reproduces the write-up as printed, and `stoichiometry="balanced"` is the
default and the one every result here uses. **The sequential document still
prints the lumped form its own MATLAB does not use**, which is why the switch
is still needed. `docs/ISSUES.md` items 5, 18 and 37.

In [ ]:
# The same run started copper-free, both stoichiometries, side by side.
y0_free = [0.0, 1e4, 1e5, demo["n_Au0"], 0.0, demo["n_Cu0"], 0.0, 0.0, 0.0]

for how in ("source", "balanced"):
    _, trx = leach.simulate_stage3(demo, 1e4, 1e5, t_end=200.0, y0=y0_free,
                                   model="B", stoichiometry=how)
    print(f"{how:<9}  Cu(II) min {trx['Cu_II'].min():+10.4f}   "
          f"S4O6 min {trx['S4O6'].min():+12.4f} umol/L")

print()
print("The write-up as printed drives Cu(II) negative. Charging each pathway")
print("its own coefficients does not. Tetrathionate stays positive here: an")
print("earlier version of this check showed it going negative too, but that")
print("came from an invented enzyme rate about 640x too large.")
print("ISSUES.md items 5, 18 and 37.")

---

# The three vessels, run end to end

A one-pot design was also written and was retired on 2026-08-12. Its write-up
and MATLAB stay in `docs/source/sequential_2026-08-12/`, marked retired, and
nothing here runs it.

The conversion vessel has no cells, so the enzyme is an input rather than a
state. It has no copper either, so nothing can destroy thiosulfate and the whole
broth charge banks. The leach vessel has no enzyme, so thiosulfate is a fixed
budget.

It was adopted for `docs/ISSUES.md` item 32, not for its yield: TetH will not
fold actively inside *E. coli*, and only this design refolds it outside the
cell. It bought 1.9 percentage points of gold over the retired one-pot design.

Reproduced line for line before adoption. `docs/ISSUES.md` item 36.

In [ ]:
P_SEQ = leach.parameters()
run = leach.simulate_sequential(P_SEQ, n=20001)
s2, s3 = run["stage2"], run["stage3"]
i24 = int(np.argmin(abs(s3["t"] - 1440)))

print("STAGE 2  conversion vessel, no copper present")
print(f"  tetrathionate  {P_SEQ['S4O6_initial']:8.0f} -> {s2['S4O6'][-1]:7.0f} umol/L")
print(f"  thiosulfate    {P_SEQ['S2O3_initial']:8.0f} -> {s2['S2O3'][-1]:7.0f} umol/L")
print(f"  conversion     {100 * (1 - s2['S4O6'][-1] / P_SEQ['S4O6_initial']):8.1f} %")
print(f"  sulfur closure {abs(s2['S4O6'][-1] + s2['S2O3'][-1] - P_SEQ['S4O6_initial'] - P_SEQ['S2O3_initial']):8.1e} umol/L")

print()
print("STAGE 3  leach vessel, thiosulfate is a budget")
print(f"  gold at 24 h   {100 * s3['gold_recovered'][i24]:8.1f} %")
print(f"  gold at 72 h   {100 * s3['gold_recovered'][-1]:8.1f} %")
print(f"  copper at 72 h {100 * s3['copper_recovered'][-1]:8.3f} %")
print(f"  Cu(II) peak    {s3['Cu_II'].max():8.0f} umol/L")
print(f"  S2O3 left      {max(s3['S2O3'][-1], 0.0):8.1f} umol/L")

ledger = leach.thiosulfate_ledger(P_SEQ, s3["t"], s3)
print()
print("WHERE THE LEACHANT WENT")
for name in ("gold", "copper", "parasitic"):
    print(f"  {name:<10} {ledger[name]:9.0f} umol/L  {100 * ledger[name] / ledger['total']:6.2f} %")
print(f"  {'total':<10} {ledger['total']:9.0f} umol/L")
print("The leachant is not consumed by leaching. It is consumed by Cu(II).")

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(14, 3.2))
th2, th3 = s2["t"] / 60, s3["t"] / 60

ax[0].plot(th2, s2["S4O6"], label="S4O6")
ax[0].plot(th2, s2["S2O3"], label="S2O3")
ax[0].set_title("(a) conversion vessel, closed")
ax[0].set_xlabel("time (h)"); ax[0].set_ylabel("umol/L"); ax[0].legend()

ax[1].plot(th3, s3["S4O6"], label="S4O6 (dead end)")
ax[1].plot(th3, s3["S2O3"], label="S2O3 (budget)")
ax[1].set_title("(b) leach vessel, sulfur")
ax[1].set_xlabel("time (h)"); ax[1].legend()

ax[2].plot(th3, s3["Cu_II"], label="Cu(II), oxidant")
ax[2].plot(th3, s3["Cu_I"], "--", label="Cu(I), inactive")
ax[2].set_title("(c) leach vessel, copper redox")
ax[2].set_xlabel("time (h)"); ax[2].legend()

ax[3].plot(th3, 100 * s3["gold_recovered"], label="Au")
ax[3].plot(th3, 100 * s3["copper_recovered"], "--", label="Cu")
ax[3].set_title("(d) Recovery")
ax[3].set_xlabel("time (h)"); ax[3].set_ylabel("%"); ax[3].legend()

for a in ax:
    a.grid(alpha=0.3)
fig.tight_layout()

### The reaction nobody was watching. `docs/ISSUES.md` item 41

Both write-ups name copper(II) as the villain, and the ledger above agrees:
98% of the leachant goes down the parasitic reaction with Cu(II).

But copper metal does **not** dissolve as Cu(II). It dissolves as Cu(I), on
both routes. There is exactly one reaction anywhere in the leaching model that
makes Cu(II), and it is oxygen:

`4 Cu(I) + O2 + 2 H2O -> 4 Cu(II) + 4 OH-`, at rate `k_ox_Cu * [O2] * [Cu(I)]`

So the poison is not manufactured by the board. It is manufactured by the
sparger, and `[O2]` is an operating choice. Oxygen drives the leaching routes
and it makes the poison, so recovery has an interior maximum in `[O2]`.

In [ ]:
o2_levels = [50, 60, 80, 100, 150, 250, 500, 1000]
rec = [100 * leach.simulate_sequential(dict(P_SEQ, O2=float(o)), n=801)["stage3"]["gold_recovered"][-1]
       for o in o2_levels]

no_reox = leach.simulate_sequential(dict(P_SEQ, k_ox_Cu=0.0), n=801)["stage3"]
print("With Cu(I) re-oxidation switched off entirely (not physical, but telling):")
print(f"  Cu(II) peak    {no_reox['Cu_II'].max():8.2f} umol/L   (baseline {s3['Cu_II'].max():.0f})")
print(f"  S2O3 left      {no_reox['S2O3'][-1]:8.0f} umol/L   (baseline 0)")
print(f"  gold at 72 h   {100 * no_reox['gold_recovered'][-1]:8.1f} %        (baseline {100 * s3['gold_recovered'][-1]:.1f})")

best = int(np.argmax(rec))
print()
print(f"Best of the oxygen levels tested: {o2_levels[best]} umol/L -> {rec[best]:.1f} % gold")
print(f"Baseline 500 umol/L             -> {rec[o2_levels.index(500)]:.1f} % gold")
print("Turning the sparger down is the cheapest change available.")

fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.plot(o2_levels, rec, "o-")
ax.axvline(500, color="k", ls=":", lw=1)
ax.annotate("baseline", (500, min(rec)), textcoords="offset points", xytext=(6, 4))
ax.set_xscale("log")
ax.set_xlabel("dissolved oxygen (umol/L)")
ax.set_ylabel("gold recovered at 72 h (%)")
ax.set_title("Oxygen: oxidant below the peak, poison above it")
ax.grid(alpha=0.3)
fig.tight_layout()

**What is solid and what is not.** The existence of the optimum survives every
value of `k_par` tested, though its position moves. The number 80 does not: it
rests on `k_par` and `k_ox_Cu`, both borrowed from ammoniacal literature where
ammonia deliberately stabilises Cu(II). Do not put it on a set-point sheet.

There is also no oxygen mass-transfer term anywhere in the leaching model.
`[O2]` is held, not solved, and running deliberately oxygen-poor is the regime
where a held `[O2]` is least defensible.

---

# Handover 1 — leachate gold becomes `Au_feed`

The recovery-chain document §2.1 is unambiguous about what this number does:
**`Au_feed` is the dominant design variable**, and the capture transition sits
exactly at the capacity ceiling $q_{max}X_{max}$.

That ceiling was 22 µM when the document was written, at the retired
`q_max` = 4.4. At the adopted `q_max` it is **0.457 µM**, and the leach delivers
29 µM. Dilution is therefore a legitimate design lever, and the adapter reports
where a given feed lands rather than silently clipping it. `docs/ISSUES.md`
items 43 and 44.

### The two ligands, side by side

v5 runs Stage 3 with ammonia and then with no ligand at all. "No ligand" is
not a choice anybody proposed, so the second column here is **glycine**.

Both run at one operating point, so every difference is the ligand: pH 10.0
held by the dosing pump, oxygen from the air pump, 6 mm crush, 72 h. Glycine
is reported as a **band**, because `eps_gly` is unmeasured and spans 17x.

The comparison is deliberately generous to ammonia. It is assumed not to be
stripped by the air pump, though it would be; it keeps its own measured
constants; and `k_ox_Cu` is ammoniacal in both columns. ISSUES.md item 59.


In [ ]:
L = leach.compare_ligands(n=2001)
ROWS = [("ammonia", "ammonia 266 mM"),
        ("glycine_low", "glycine eps 0.0024"),
        ("glycine_mid", "glycine eps 0.0116"),
        ("glycine_high", "glycine eps 0.0421")]

print(f"{'':22s}{'gold 24h':>10s}{'gold 72h':>10s}{'S2O3 end':>10s}"
      f"{'polythionate':>14s}{'titrant':>10s}")
for key, name in ROWS:
    r = L[key][1]
    print(f"{name:22s}{r['gold_24h']:9.2f} %{r['gold_72h']:9.2f} %"
          f"{r['S2O3_end']:10.0f}{r['S4O6_end'] + r['S3O6_end']:14.0f}"
          f"{r['titrant']:10.0f}")

amm = L["ammonia"][1]
mid = L["glycine_mid"][1]
print()
print("Ammonia wins the gold column: 100.0 % against 83.8 % at 72 h.")
print("It does NOT win at 24 h, where the gap is under two points, and v5's")
print("own recommendation is a one-day batch, not three.")
print()
print(f"Polythionate handed to capture: {amm['S4O6_end'] + amm['S3O6_end']:.0f}"
      f" with ammonia, {mid['S4O6_end'] + mid['S3O6_end']:.0f} with glycine.")
print("ISSUES.md item 44 is BLOCKING and it blocks on exactly that number.")
print("Reaching 100 % gold is worth nothing if the capture stage cannot take it.")
print()
print("Every ledger closes and every sulfur audit holds:")
for key, name in ROWS:
    p, r = L[key]
    led = r["ledger"]
    close = abs(led["total"] - led["actually_lost"]) / abs(led["actually_lost"])
    print(f"  {name:22s} ledger {close:8.1e}   sulfur atoms {r['sulfur_drift']:8.1e}")

print()
print("And the 98 % figure needs qualifying once the alkaline route is open:")
for key, name in ROWS:
    led = L[key][1]["ledger"]
    print(f"  {name:22s} gross destroyed {led['gross_destroyed']:8.0f}"
          f"   returned {led['returned']:8.0f}   net {led['net_destroyed']:8.0f}")
print("Thiosulfate the alkaline leg hands back can be destroyed again, so the")
print("gross figure double-counts. Net is what the process actually consumed.")


In [ ]:
au_feed = interfaces.au_feed_from_leach(tr["Au_complex"][-1], unit="umol/L")
print(f"Au_feed from the leach = {au_feed:.3f} uM")
print("guidance:", interfaces.feed_warning(au_feed) or "inside the 10-20 uM window")

print()
for feed in (3, 10, 15, 22, 25, 50, 300):
    note = interfaces.feed_warning(feed)
    print(f"  {feed:>4} uM  {'OK' if note is None else note[:88]}")

print()
cu = interfaces.cu_feed_from_leach(tr["Cu_I"][-1], tr["Cu_II"][-1])
print("copper carried into capture:", {k: round(v, 3) if isinstance(v, float) else v
                                        for k, v in cu.items()})
print("The recovery chain has no copper term. Selectivity against Cu/Ag/Ni is unknown "
      "(model.md section 0).")

In [ ]:
# The number that actually crosses the joint, from the sequential leach.
seq_feed = interfaces.au_feed_from_leach(s3["Au_complex"][-1], unit="umol/L")
print(f"sequential leach, 72 h  ->  Au_feed = {seq_feed:6.1f} uM")
print("   ", interfaces.feed_warning(seq_feed) or "inside the window")

print()
print("And every improvement to the leach makes the handover worse, because the")
print("recovered gold and the feed concentration are the same number:")
for label, over in (("oxygen at the optimum, 80 uM", dict(O2=80.0)),
                    ("thinnest plating, t_film 0.05 um", dict(t_film=0.05e-6)),
                    ("enzyme carried into stage 3", dict(carry_enzyme=True))):
    alt = leach.simulate_sequential(dict(P_SEQ, **over), n=801)["stage3"]
    print(f"  {label:<34} gold {100*alt['gold_recovered'][-1]:5.1f} %"
          f"   feed {alt['Au_complex'][-1]:6.1f} uM")

print()
print("Optimising the leach alone drives the capture off its cliff.")
print("ISSUES.md item 39, and it needs the stopping rule of item 21.")
print()
print("=" * 72)
print("AND THE SULFUR, WHICH IS THREE ORDERS OF MAGNITUDE LARGER THAN THE GOLD")
print("=" * 72)
sulf = interfaces.sulfur_feed_from_leach(
    s4o6=float(s3["S4O6"][-1]), s2o3=float(s3["S2O3"][-1]),
    s3o6=float(s3["S3O6"][-1]))
print(f"  tetrathionate S4O6  {sulf['S4O6_uM']:9.0f} uM")
print(f"  trithionate   S3O6  {sulf['S3O6_uM']:9.0f} uM")
print(f"  thiosulfate   S2O3  {sulf['S2O3_uM']:9.0f} uM")
print(f"  polythionate total  {sulf['polythionate_uM']:9.0f} uM")
print()
print(interfaces.thiol_blocking_warning(sulf["S4O6_uM"],
                                        s3o6_uM=sulf["S3O6_uM"]))

print()
print("Under the two ligand scenarios, where the alkaline route is running:")
for key, name in (("ammonia", "ammonia"), ("glycine_mid", "glycine")):
    t3 = L[key][1]["traj"]
    f = interfaces.sulfur_feed_from_leach(s4o6=float(t3["S4O6"][-1]),
                                          s2o3=float(t3["S2O3"][-1]),
                                          s3o6=float(t3["S3O6"][-1]))
    thiols = (interfaces.THIOLS_PER_GOLB_SITE
              * P_CHAIN["V1_capture"]["q_max"]["value"]
              * P_CHAIN["V1_capture"]["X_max"]["value"])
    print(f"  {name:8s} S4O6 {f['S4O6_uM']:7.0f}  S3O6 {f['S3O6_uM']:7.0f}"
          f"  ->  {f['S4O6_uM']/thiols:9,.0f} : 1 measured,"
          f"  {f['polythionate_uM']/thiols:9,.0f} : 1 bound")
print()
print("The second column is an UPPER BOUND. Trithionate's reactivity toward")
print("thiols relative to tetrathionate is not published, so it is counted")
print("as if equal. ISSUES.md items 44 and 60.")


---

# GolB display — surface site density

*Full text: `docs/02_golb_expression.md`. Code: `src/expression.py`.*

Four pools in molecules per cell, each fed from upstream and drained by
degradation and by growth dilution:

$$m \;\rightarrow\; P_c \;\rightarrow\; P_s \;\rightarrow\; A = f P_s$$

$$\frac{dm}{dt} = \alpha N - (\delta_m + \mu)m$$
$$\frac{dP_c}{dt} = \beta m - (k_{sec} + \delta_c + \mu)P_c$$
$$\frac{dP_s}{dt} = k_{sec}P_c\left(1 - \frac{P_s}{P_{s,max}}\right) - (\delta_s + \mu)P_s$$

Dilution appears because a growing cell doubles its volume; even a perfectly
stable protein is diluted at the growth rate. For mRNA that term is small, for
stable protein it dominates.

In [ ]:
t_e, m, Pc, Ps, A = expression.simulate(P_EXPR, t_end=400.0)
ss = expression.steady_state(P_EXPR)
reported = params.load("expression")["reported_steady_state"]

print(f"{'':<8}{'ODE':>10}{'closed form':>14}{'reported':>10}")
for key, series in (("m", m), ("P_c", Pc), ("P_s", Ps), ("A", A)):
    print(f"{key:<8}{series[-1]:>10.1f}{ss[key]:>14.1f}{reported[key]:>10}")

gap = 100.0 * abs(ss["P_s"] - Ps[-1]) / Ps[-1]
print(f"\nODE and closed form agree to {gap:.1e} %. The closed form KEEPS the")
print("saturation bracket since draft 4, so there is one calculation and not")
print("two. The 5.5% gap of ISSUES.md item 3 was the non-saturating form at")
print("N = 20; item 3 is closed. See item 61.")
lin = expression.steady_state_linear(P_EXPR)
print(f"Dropping the bracket at N = {P_EXPR['N']:.0f} would overpredict P_s by "
      f"{100.0 * (lin['P_s'] / ss['P_s'] - 1.0):.0f} %.")

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(10, 6.4))

ax[0, 0].plot(t_e, m)
ax[0, 0].set(xlabel="time (min)", ylabel="mRNA / cell", title="(a) transcript dynamics")

ax[0, 1].plot(t_e, Pc, label="cytoplasmic $P_c$")
ax[0, 1].plot(t_e, Ps, label="surface $P_s$")
ax[0, 1].plot(t_e, A, label="active sites $A$")
ax[0, 1].set(xlabel="time (min)", ylabel="molecules / cell",
             title="(b) protein and active sites")
ax[0, 1].legend()

k_range = np.logspace(-3, 1, 200)
sites = [expression.steady_state({**P_EXPR, "k_sec": k})["A"] for k in k_range]
ax[1, 0].semilogx(k_range, sites)
ax[1, 0].plot(P_EXPR["k_sec"], ss["A"], "o", ms=7, label="operating point")
ax[1, 0].set(xlabel="$k_{sec}$  (min$^{-1}$)", ylabel="steady-state sites / cell",
             title="(c) sensitivity to display rate")
ax[1, 0].legend()

symbols = ["alpha", "N", "beta", "f", "k_sec", "delta_c", "delta_s", "delta_m", "mu"]
sens = [expression.normalised_sensitivity(P_EXPR, s) for s in symbols]
ax[1, 1].bar(range(len(symbols)), sens,
             color=["tab:green" if v > 0 else "tab:red" for v in sens])
ax[1, 1].set_xticks(range(len(symbols)))
ax[1, 1].set_xticklabels(symbols, rotation=45, ha="right")
ax[1, 1].axhline(0, color="k", lw=0.8)
ax[1, 1].set(ylabel="normalised sensitivity of $A^*$",
             title="(d) which parameter moves site density")

fig.tight_layout()

**Reading panel (d).** Promoter strength $\alpha$, copy number $N$, translation
strength $\beta$ and functional fraction $f$ all sit near +1: a 1% increase in
any of them gives about 1% more sites. These are the engineering handles.
$k_{sec}$ is only about +0.2, because it appears in both a numerator and a
denominator — which is the same story panel (c) tells. Growth rate $\mu$ is the
strongest negative influence, appearing in all three loss terms.

**The practical message is that making secretion faster yields diminishing
returns.** Raise transcription, translation, or copy number instead.

### Coupling to gold uptake

Each active site binds one Au(I) through a single Cys-X-X-Cys motif, so the
Langmuir plateau is the site count itself:

$$q(C) = \frac{A\,C}{K_Au + C}$$

**Note the `K_Au` conflict.** The display document uses 10 µM as an explicit placeholder;
the recovery chain uses the 0.7 µM GolB-based literature proxy. The half-saturation point of the figure below is `K_Au` by
construction, so the whole x-axis shifts 100× depending on which is right.
`docs/ISSUES.md` item 1.

In [ ]:
C = np.logspace(-2, 3, 300)
A_star = ss["A"]

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for K_Au, panel, title in ((10.0, ax[0], "$K_Au$ = 10 uM  (display document)"),
                          (0.7, ax[1], "$K_Au$ = 0.7 uM  (S78 literature proxy)")):
    for mult, label in ((1, "base"), (2, "2x"), (5, "5x")):
        panel.semilogx(C, expression.langmuir_uptake(C, A_star * mult, K_Au),
                       label=f"{label} ({A_star*mult:.0f} sites)")
    panel.axvline(K_Au, color="k", ls=":", lw=1)
    panel.set(xlabel="leachate Au(I)  (uM)", title=title)
ax[0].set_ylabel("gold ions bound / cell")
ax[0].legend(fontsize=8)
fig.suptitle("Langmuir uptake — the same model under the two published $K_Au$ values", y=1.05)
fig.tight_layout()

---

# Handover 2 — active sites become `q_max`

$$q_{max}\;\left[\frac{\mu M}{g/L}\right] = A\,[\text{sites/cell}] \times \frac{\text{cells}}{\text{g}} \times \frac{10^{6}}{N_A}$$

**This handover does not close, and the gap is the most important finding in
this repository.**

In [ ]:
q_from_II = interfaces.qmax_from_expression(A_star)
report = interfaces.qmax_disagreement(q_from_II)

print(f"display cascade  ->  q_max = {report['part_II_uM_per_gL']:.4f} uM/(g/L)")
print(f"recovery chain   q_max = {report['part_III_uM_per_gL']:.4f} uM/(g/L)"
      f"  (retired 2026-08-10, kept for the comparison)")
print(f"recovery chain   q_max = "
      f"{P_CHAIN['V1_capture']['q_max']['value']:.4f} uM/(g/L)  (in use)")
print(f"recovery sweep low = 0.33 uM/(g/L)")
print(f"\nThe recovery value is {1/report['ratio_II_over_III']:.0f}x higher than the cascade,")
print(f"and {0.33/q_from_II:.0f}x higher than the bottom of its own sweep.")

X_max = P_CHAIN["V1_capture"]["X_max"]["value"]
print(f"\ncapacity ceiling q_max * X_max at X_max = {X_max} g/L:")
print(f"   using retired 4.4    ->  {4.4*X_max:8.2f} uM   (model.md section 2.1 quotes 22 uM)")
print(f"   using cascade  {q_from_II:.4f} ->  {q_from_II*X_max:8.4f} uM")
print("\nThe recommended window was revised to 0.4-0.8 uM on 2026-08-11. Since")
print("2026-08-19 the in-use ceiling lands INSIDE that window rather than below")
print("all of it, so a 0.4 uM feed is the first one in the window that is not")
print("capacity-limited. ISSUES.md items 2, 61 and 62.")

# Recovery chain: conditional scenarios

The current technical review is [RECOVERY.md](docs/RECOVERY.md). Capture,
elution and reduction form the chain; physical collection is excluded.
The reference calculation and full kinetic range are scenarios, not a validated
mean or confidence interval. Historical ensemble claims are recorded in ISSUES.

Elution curves below compare empirical candidate laws. Their common numerical
rate is an illustration, not a parameter fitted jointly to both laws.


In [ ]:
phi = P_CHAIN["elution"]["phi"]["value"]
k_elute = P_CHAIN["elution"]["k_elute"]["value"]

t_k = np.linspace(0, 12, 400)
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(t_k, recovery_chain.eta_elute(t_k, phi, k_elute),
        lw=2, label="PFO, in use")
ax.plot(t_k, recovery_chain.eta_elute_pso(t_k, phi, k_elute),
        lw=1.2, ls="-.", label="PSO, retired comparison")
ax.axhline(1 - phi, color="k", ls="--", lw=1,
           label=f"ceiling $1-\phi$ = {1-phi:.2f}")
ax.axvline(3 / k_elute, color="grey", ls=":", lw=1, alpha=0.5,
           label="$3/k$, 95% (PFO)")
ax.set(xlabel="elution time $t_k$ (h)", ylabel="$\eta_{elute}$",
       title=f"First-order elution at k = {k_elute} /h")
ax.legend(fontsize=8)
fig.tight_layout()

for t in (1.5, 3.0, 6.0, 9.5):
    print(f"t = {t:>4} h   PFO {recovery_chain.eta_elute(t, phi, k_elute):.3f}"
          f"   PSO comparison {recovery_chain.eta_elute_pso(t, phi, k_elute):.3f}")
print("")
print("The S60 acid-thiourea condition is the working eluent; k_elute remains provisional for GolB.")

## Electron inventory and stage volumes

Electron supply is a necessary ceiling, not proof of conversion. The report
assumes equal stage volumes and separately supplied protein. Use actual
transferred amounts and receiving volumes when specifying a real process.


In [ ]:
from src import recovery_audit
p_recovery = recovery_audit.operating_parameters()
print("Protein target (uM):", p_recovery["E_tot"])
print("Dithionite charge (uM):", p_recovery["Rech_0"])
print("These are scenario inputs, not measured active protein or conversion.")


## Recomputed recovery result

Every stage is evaluated from YAML and the display interface. The slow/fast
kinetic corners are explicit scenarios, not a confidence interval.


In [ ]:
reference = recovery_audit.stage_amounts()
for name, value in reference.items():
    print(name, value)
print("Kinetic scenarios:", recovery_audit.reduction_scenario_range())
print("Metal formed is reported per initial litre; collection is excluded.")


# What to measure next

Use the priorities and controls in [RECOVERY.md](docs/RECOVERY.md): binding
capacity in leachate; calibrated reduction time courses with product confirmation;
elution at actual loading; active soluble protein over culture time.

The revised fitter does not turn an optical plateau into full conversion.
Protein dilution follows actual growth; stationary synthesis and decay still
need calibration. Capture damage coupling remains null. There is no validated
harvest optimum or demonstrated complete reduction at present.

Upstream activity, matrix compatibility, plating thickness and copper/oxygen
chemistry remain separate uncertainties. This notebook's synthetic and
literature comparisons are verification and scenarios, not the team's wet-lab data.


## Recovery amounts and reduction trajectories (2026-09-07)
The technical report is `docs/RECOVERY_MODEL.docx`. The explicit process interface preserves gold amounts across volumes and excludes seed from new metal. The reference protein dose is a supplied target; lysate mode requires a protein concentration. The returned trajectory includes gold and reducing-capacity balance residuals.

In [ ]:
from src import recovery_process
recovery_result = recovery_process.run()
print({name: recovery_result[name] for name in ("feed_gold_amount_umol", "new_metal_amount_umol", "gold_balance_residual_umol")})
reduction_trace = recovery_result["reduction_trajectory"]
print("Largest electron ledger residual (uM):", abs(reduction_trace["electron_residual"]).max())
